In [4]:
using ITensors, ITensorMPS


# Matrix Product State/Tree Tensor Network

![TT1](./TT1.png)


$$ T^{s_1 s_2 s_3 s_4 s_5 s_6}=\sum_{\{\alpha\}} A_{\alpha_1}^{s_1} A_{\alpha_1 \alpha_2}^{s_2} A_{\alpha_2 \alpha_3}^{s_3} A_{\alpha_3 \alpha_4}^{s_4} A_{\alpha_4 \alpha_5}^{s_5} A_{\alpha_5}^{s_6} $$

where, $s_i$'s are physical bonds and $\alpha_i$'s are virtual bond. Virtual bond dimension $m=d^{N/2}$ can express any tensor of N physical bonds each with $d$-dimensions exactly. In most applications m can be much smaller.

Total parameters for the tensor $d^N$. With an MPS of VBD of $m$, total paramters  $N(dm^2)$.

MPS reprentation is independent of N, by assuming all the factor tensors $A_i$ are the same by TT gauge transformations.

In [3]:
d = 2
N = 3
# Julia function to generate 'random number from normal
# distribution with mean 0 and var 1/2.
T = randn(fill(d,N)...) #splat operator to unpack vector element
for i in 1:d
    for j in 1:d
            println(T[:,:,j])
    end
end


sites = siteinds(d,N)
cutoff = 1E-8
maxdim = 10   # bond dimension

#convert the random array into MPS
M = MPS(T,sites;cutoff=cutoff,maxdim=maxdim)



[-0.02790652663388138 0.7391907952288814; -0.6022712839251197 -1.1348171009660983]
[0.661818279310417 1.195788238740864; -0.41713324868849255 0.5747564691855293]
[-0.02790652663388138 0.7391907952288814; -0.6022712839251197 -1.1348171009660983]
[0.661818279310417 1.195788238740864; -0.41713324868849255 0.5747564691855293]


MPS
[1] ((dim=2|id=669|"Site,n=1"), (dim=2|id=989|"Link,n=1"))
[2] ((dim=2|id=989|"Link,n=1"), (dim=2|id=454|"Site,n=2"), (dim=2|id=475|"Link,n=2"))
[3] ((dim=2|id=475|"Link,n=2"), (dim=2|id=403|"Site,n=3"))


# Retrieving a single component of MPS
![TT2](./TT2.png)

Cost is $Nm^2$

In [4]:
N = 10
s = siteinds(2,N)
chi = 4
psi = randomMPS(s;linkdims=chi)

# Make an array of integers of the element we
# want to obtain
el = [1,2,1,1,2,1,2,2,2,1]

V = ITensor(1.)
for j=1:N
  V *= (psi[j]*state(s[j],el[j]))
end
v = scalar(V)

# v is the element we wanted to obtain:
@show v

v = 0.008187170979239282


0.008187170979239282

In [5]:
@show M[1]
println(N)
s = siteinds(2,N)
el=[1,1,2]
V = ITensor(1.)
for i=1:N
    V = (M[i] * state(s[i],el[i]) )
    
    @show V
end



M[1] = ITensor ord=2
Dim 1: (dim=2|id=669|"Site,n=1")
Dim 2: (dim=2|id=989|"Link,n=1")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2
 -0.8068200107262383  -0.59079731743781
  0.59079731743781    -0.8068200107262383
10
V = ITensor ord=3
Dim 1: (dim=2|id=669|"Site,n=1")
Dim 2: (dim=2|id=989|"Link,n=1")
Dim 3: (dim=2|id=969|"Site,n=1")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2×2
[:, :, 1] =
 -0.8068200107262383  -0.59079731743781
  0.59079731743781    -0.8068200107262383

[:, :, 2] =
 0.0  0.0
 0.0  0.0
V = ITensor ord=4
Dim 1: (dim=2|id=989|"Link,n=1")
Dim 2: (dim=2|id=454|"Site,n=2")
Dim 3: (dim=2|id=475|"Link,n=2")
Dim 4: (dim=2|id=701|"Site,n=2")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2×2×2
[:, :, 1, 1] =
  0.4898506746199167   0.7768647709052002
 -0.16793885158562422  0.35822337504541035

[:, :, 2, 1] =
 0.1795767911431079   -0.4277074134620277
 0.31304194350469666   0.8287480233214248

[:, :, 1, 2] =
 0.0  0.0
 0.0  0.0

[:, :, 2, 2] =
 0.0  0.0
 0.0  0.0
V = ITensor ord

LoadError: BoundsError: attempt to access 3-element Vector{ITensor} at index [4]

# How to create MPS form of the following wavepackets

$$|k= \pm \pi / 2\rangle=\frac{1}{2}(|0001\rangle \pm i|0010\rangle-|0100\rangle \mp i|1000\rangle) $$
$$|k= \pi\rangle=\frac{1}{2}(|0001\rangle -|0010\rangle+|0100\rangle - |1000\rangle) $$



In [34]:
N=16

function gaussian_wavepacket(x, x0, k0, sigma)
    return exp(-0.5 * ((x - x0) / sigma)^2 + im * k0 * x)
end
             # Number of lattice sites
x0 = N/4            # Center of the wave packet
k0 = π / N            # Initial momentum
sigma = N / 8        # Width of the wave packet

# Create a 1D lattice
sites = siteinds("S=1/2", N)

wavepacket_value=zeros(Complex{Float64},N)
# Set the wave packet values
for j in 1:N
    wavepacket_value[j] = gaussian_wavepacket(j, x0, k0, sigma)
end

ITensors.state(::StateName"Emp", ::SiteType"Electron") = [1.0, 0, 0, 0]
ITensors.state(::StateName"Up", ::SiteType"Electron") = [0.0, 1, 0, 0]
ITensors.state(::StateName"Dn", ::SiteType"Electron") = [0.0, 0, 1, 0]
ITensors.state(::StateName"UpDn", ::SiteType"Electron") = [0.0, 0, 0, 1]

ITensors.state(::StateName"+", ::SiteType"Electron") = [0, 1/sqrt(2), 1/sqrt(2), 0]
s = siteind("Electron")
plus = state("+",s)
@show plus

plus = ITensor ord=1
Dim 1: (dim=4|id=32|"Electron,Site")
NDTensors.Dense{Float64, Vector{Float64}}
 4-element
 0.0
 0.7071067811865475
 0.7071067811865475
 0.0


ITensor ord=1 (dim=4|id=32|"Electron,Site")
NDTensors.Dense{Float64, Vector{Float64}}

In [55]:
s = Index(3,"Site,S=1")
Sz = op("Sz", s)
@show Sz

sites = siteinds("S=1/2",3)
M = [1/2 0 ; 0 -1/2]
Sz2 = op(M,sites[2])
println(Sz2)

ITensors.op(::OpName"Pup",::SiteType"S=1/2") =[1 0; 0 0]
    
s = siteinds("S=1/2",N)
Pup1 = op("Pup",s[1])
Pup3 = op("Pup",s[3])

@show Pup1

Sz = ITensor ord=2
Dim 1: (dim=3|id=502|"S=1,Site")'
Dim 2: (dim=3|id=502|"S=1,Site")
NDTensors.Dense{Float64, Vector{Float64}}
 3×3
 1.0  0.0   0.0
 0.0  0.0   0.0
 0.0  0.0  -1.0
ITensor ord=2
Dim 1: (dim=2|id=12|"S=1/2,Site,n=2")'
Dim 2: (dim=2|id=12|"S=1/2,Site,n=2")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2
 0.5   0.0
 0.0  -0.5
Pup1 = ITensor ord=2
Dim 1: (dim=2|id=368|"S=1/2,Site,n=1")'
Dim 2: (dim=2|id=368|"S=1/2,Site,n=1")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2
 1.0  0.0
 0.0  0.0


ITensor ord=2 (dim=2|id=368|"S=1/2,Site,n=1")' (dim=2|id=368|"S=1/2,Site,n=1")
NDTensors.Dense{Float64, Vector{Float64}}

In [66]:
N=4
s = siteinds("S=1/2",N)
state1 = ["Up" for n=1:N]
state2 = [isodd(n) ? "Up" : "Dn" for n=1:N]
psi1=MPS(s,state1)
psi2=MPS(s,state2)
psi_c=0.5im*psi1+0.5*psi2
normalize!(psi_c)
println(psi_c)

MPS
[1] ((dim=2|id=776|"S=1/2,Site,n=1"), (dim=2|id=983|"Link,l=1"))
[2] ((dim=2|id=627|"S=1/2,Site,n=2"), (dim=2|id=447|"Link,l=2"), (dim=2|id=983|"Link,l=1"))
[3] ((dim=2|id=652|"S=1/2,Site,n=3"), (dim=2|id=871|"Link,l=3"), (dim=2|id=447|"Link,l=2"))
[4] ((dim=2|id=607|"S=1/2,Site,n=4"), (dim=2|id=871|"Link,l=3"))



In [131]:
let

    
    # Define a function to create a Gaussian wave packet
    function gaussian_wavepacket(x, x0, k0, sigma)
        return exp(-0.5 * ((x - x0) / sigma)^2 + im * k0 * x)
    end

    function create_wavepacket_MPS(sites,N,x0,k0,sigma,subindex_arr,want_normalize=1)
        
        # Initialize the MPS for the wave packet
        psi_c = MPS(sites)
        # Set the wave packet values
        for j in subindex_arr
            wavepacket_value = gaussian_wavepacket(j, x0, k0, sigma)
            println(wavepacket_value)
            state = [k==j ? "Up" : "Dn" for k=1:N]
            if j==1
                psi_c= wavepacket_value * MPS(sites,state)
            else
                psi_c += (wavepacket_value * MPS(sites,state) )
            end
        end
        # Normalize the wave packet
        if want_normalize==1
            normalize!(psi_c)
        end
        
        return psi_c
    end

     # Parameters for the wave packet
    N = 16              # Number of lattice sites
    x0 = N/4            # Center of the wave packet
    k0 = π / N            # Initial momentum
    sigma = N / 8        # Width of the wave packet
    # Create a 1D lattice
    sites = siteinds("S=1/2", N)
    mysubindex_arr=collect(1:2:N)
    println("odd_index",mysubindex_arr)
    psi_kplus=create_wavepacket_MPS(sites,N,x0,k0,sigma,mysubindex_arr,0)
    # Print the MPS
    #println("kplus:")
    #@show psi_kplus.data

    println("check")
    for excite_id=1:N
        el = fill(2,N)
        el[excite_id]=1
    
        V = ITensor(1.)
        for j=1:N
          V *= (psi_kplus[j]*state(sites[j],el[j]))
        end

        #println(V)
        v = scalar(V)
        println(v)
    end
    
end

odd_index[1, 3, 5, 7, 9, 11, 13, 15]
0.31841436123165967 + 0.06333655440027101im
0.7337693574502107 + 0.4902890098080009im
0.49028900980800094 + 0.7337693574502107im
0.06333655440027104 + 0.31841436123165967im
-0.008571670528991803 + 0.04309269776389177im
-0.0012153049502571752 + 0.0018188323919507848im
-3.331307729010321e-5 + 2.2259086608601484e-5im
-2.647706859391427e-7 + 5.2666163952890895e-8im
check
0.3184143612316597 + 0.06333655440027094im
-1.1071497298385556e-15 + 3.2627478970506354e-16im
0.7337693574502105 + 0.49028900980800183im
-1.8346363291908964e-16 + 4.565503997470841e-16im
0.49028900980800033 + 0.7337693574502097im
1.2498861905203486e-47 + 1.6201408205102091e-47im
0.06333655440027099 + 0.3184143612316594im
0.0 + 0.0im
-0.008571670528991784 + 0.04309269776389174im
0.0 + 0.0im
-0.0012153049502571743 + 0.0018188323919507835im
0.0 + 0.0im
-3.33130772901032e-5 + 2.2259086608601494e-5im
0.0 + 0.0im
-2.647706859391427e-7 + 5.2666163952890895e-8im
-0.0 + 0.0im


In [10]:
using ITensors
let
  # ... your own code goes here ...
  # For example:
  i = Index(2,"i")
  j = Index(2,"j")
  T = random_itensor(i,j)
  @show T
end

T = ITensor ord=2
Dim 1: (dim=2|id=985|"i")
Dim 2: (dim=2|id=409|"j")
NDTensors.Dense{Float64, Vector{Float64}}
 2×2
 0.18410982692391017  -0.864780907351973
 1.546878883536284    -0.6850547685539954


ITensor ord=2 (dim=2|id=985|"i") (dim=2|id=409|"j")
NDTensors.Dense{Float64, Vector{Float64}}